In [ ]:
#!/usr/bin/env sage
r"""
m5_superelliptic_pipeline_FIXED.sage
====================================
Fixed SageMath pipeline for superelliptic curves  X: y^5 = f(x)
over finite fields GF(q).

MODEL NOTE:
  The code supports two comparison models:

    * weighted/natural P(m,n,m): basis filtered by pole-weight i*m+j*n
    * classical P(1,1,1):       basis filtered by total degree i+j

  In both cases the AG/CSS distance bounds are computed from the actual pole
  order at infinity, i*m+j*n. This lets P(1,1,1) run as a genuine classical
  total-degree comparison without pretending that total degree equals pole
  degree on the superelliptic function field.

KEY FIX (vs. original):
  The original choose_css_parameters used r1 = N-1, forcing d(C1) >= 1
  and hence d_Q = 1 for every code.  This version searches the SATURATED
  CSS line  r1 + r2 = N + 2g - 2  to find the pair that maximises the
  product  k_Q * d_Q_bound, giving nontrivial designed quantum distance.

PERFORMANCE FIX:
  The search is O(N) per curve (linear scan of the saturated line with
  precomputed Riemann-Roch dimensions), not O(N^2) as in the brute-force.

For m=5, deg(f)=7: genus g = 12, semigroup <5,7>.
"""

from sage.all import *
from sage.coding.linear_code import LinearCode
import csv as _csv
import ast
import os
import sys
import time


# ═══════════════════════════════════════════════════════════════════════
#  1.  VALIDATION
# ═══════════════════════════════════════════════════════════════════════

def validate_superelliptic_curve(F, f_poly, m, weights=None):
    r"""
    Validate that  y^m = f(x)  defines a smooth superelliptic curve
    over the finite field F, and that the requested weighted projective
    model is compatible with the weighted homogenization.
    """
    result = {
        'ok': False, 'skip_reason': None,
        'n': None, 'genus': None, 'weights': None, 'model_label': None,
    }

    m = ZZ(m)
    n = ZZ(f_poly.degree())
    result['n'] = n

    if n < 2:
        result['skip_reason'] = f'deg(f)={n}<2'
        return result

    g_test = gcd(m, n)
    if g_test != 1:
        result['skip_reason'] = f'gcd({m},{n})={g_test}!=1'
        return result

    p = F.characteristic()
    if p != 0 and (m * n) % p == 0:
        result['skip_reason'] = f'char={p}|m*n={m*n}'
        return result

    if f_poly.gcd(f_poly.derivative()) != 1:
        result['skip_reason'] = 'f_not_squarefree'
        return result

    genus = ZZ((m - 1) * (n - 1)) // 2
    result['genus'] = genus

    if weights is None:
        weights = (m, n, m)

    w0, w1, w2 = [ZZ(w) for w in weights]
    result['weights'] = (w0, w1, w2)
    result['model_label'] = f'P({w0},{w1},{w2})'

    is_classical = (w0 == 1 and w1 == 1 and w2 == 1)
    if not is_classical:
        target_degree = m * w1
        if target_degree != n * w0:
            result['skip_reason'] = (
                f'weights_inadmissible: y^{m} has weighted degree {target_degree}, '
                f'but x^{n} has weighted degree {n*w0}'
            )
            return result

        for exp, coeff in f_poly.dict().items():
            if coeff == 0:
                continue
            i = exp[0] if isinstance(exp, tuple) else exp
            z_degree = target_degree - i * w0
            if z_degree < 0 or z_degree % w2 != 0:
                result['skip_reason'] = (
                    f'weights_inadmissible: term x^{i} cannot be homogenized '
                    f'to weighted degree {target_degree} with z-weight {w2}'
                )
                return result

    result['ok'] = True
    return result


# ═══════════════════════════════════════════════════════════════════════
#  2.  AFFINE RATIONAL POINTS
# ═══════════════════════════════════════════════════════════════════════

def affine_points_superelliptic(F, f_poly, m):
    r"""
    Enumerate all affine F_q-rational points (a, b) with b^m = f(a).
    """
    m = ZZ(m)
    pts = []
    for a in F:
        fa = f_poly(a)
        for b in F:
            if b ** m == fa:
                pts.append((a, b))
    return pts


# ═══════════════════════════════════════════════════════════════════════
#  3.  RIEMANN-ROCH BASIS
# ═══════════════════════════════════════════════════════════════════════

def rr_basis_superelliptic(m, n, r, weights=None):
    r"""
    Monomial basis for the selected comparison model.

    P(m,n,m):   i*m + j*n <= r
    P(1,1,1):   i + j <= r

    Distance bounds still use the actual pole order i*m+j*n.
    """
    m, n, r = ZZ(m), ZZ(n), ZZ(r)
    if weights is None:
        weights = (m, n, m)
    w0, w1, _ = [ZZ(w) for w in weights]

    basis = []
    for j in range(m):
        if j * w1 > r:
            continue
        max_i = (r - j * w1) // w0
        for i in range(int(max_i) + 1):
            basis.append((i, j))
    basis.sort(key=lambda ij: ij[0] * m + ij[1] * n)
    return basis


def pole_degree_of_basis(m, n, basis):
    """Maximum pole order at P_inf among basis monomials."""
    if not basis:
        return -1
    return max(ZZ(i) * ZZ(m) + ZZ(j) * ZZ(n) for i, j in basis)


def rr_dim(m, n, r, weights=None):
    """Dimension of the selected model basis by lattice-point counting."""
    return len(rr_basis_superelliptic(m, n, r, weights=weights))


def _precompute_model_data(m, n, weights, max_r):
    """Precompute dimensions and actual pole degrees for model degrees."""
    dims = []
    poles = []
    for r in range(max_r + 1):
        basis = rr_basis_superelliptic(m, n, r, weights=weights)
        dims.append(len(basis))
        poles.append(pole_degree_of_basis(m, n, basis))
    return dims, poles


# ═══════════════════════════════════════════════════════════════════════
#  4.  BUILD AG CODE
# ═══════════════════════════════════════════════════════════════════════

def build_weighted_ag_code(F, f_poly, m, r, pts, weights=None,
                           compute_exact_d=False):
    r"""
    Construct the evaluation AG code  C_L(D, r*P_inf).
    """
    m = ZZ(m)
    n = ZZ(f_poly.degree())
    N = len(pts)

    if N == 0:
        return {'N': 0, 'k': 0, 'd': 0, 'd_goppa': 0,
                'G_mat': None, 'basis': [], 'lc': None}

    if weights is None:
        weights = (m, n, m)

    basis = rr_basis_superelliptic(m, n, r, weights=weights)
    pole_degree = pole_degree_of_basis(m, n, basis)

    rows = []
    for (i, j) in basis:
        row = [F(a) ** i * F(b) ** j for (a, b) in pts]
        rows.append(row)

    if not rows:
        return {'N': N, 'k': 0, 'd': 0, 'd_goppa': max(0, N - pole_degree),
                'pole_degree': pole_degree, 'G_mat': matrix(F, 0, N),
                'basis': basis, 'lc': None}

    G_mat = matrix(F, rows)
    k = G_mat.rank()
    d_goppa = max(0, N - pole_degree)

    lc = None
    d = d_goppa

    if k > 0:
        pivots = G_mat.pivot_rows()
        G_red = G_mat.matrix_from_rows(pivots)
        try:
            lc = LinearCode(G_red)
            if compute_exact_d and k < 20 and N < 80:
                d = lc.minimum_distance()
            else:
                d = d_goppa
        except Exception:
            d = d_goppa

    return {
        'N': N, 'k': k, 'd': d, 'd_goppa': d_goppa,
        'pole_degree': pole_degree,
        'G_mat': G_mat, 'basis': basis, 'lc': lc,
    }


# ═══════════════════════════════════════════════════════════════════════
#  5.  CSS PARAMETER SELECTION  (FIXED — O(N) balanced search)
# ═══════════════════════════════════════════════════════════════════════

def choose_css_parameters(N, g, m, n, weights=None, strategy='product'):
    r"""
    Choose model degrees r1, r2 for the CSS construction.

    The basis is selected by model degree, but CSS validity and distance
    bounds are checked using actual pole degrees R1 and R2:
        R1 + R2 <= N + 2g - 2,
        d(C1) >= N - R1,
        d(C2^perp) >= R2 - (2g - 2).

    Strategies (all search the saturated line):
      'product'   -- maximise  k_Q * d_q_bound  (recommended)
      'balanced'  -- maximise  d_q_bound  first, then k_Q
                     (gives highest certified distance, but k_Q ~ 1)
      'max_k_q'   -- maximise  k_Q  first, then d_q_bound
                     (highest dimension, but d_q_bound ~ 1)

    OUTPUT:
      (r1, r2, d_q_bound)  or  (None, None, None)
    """
    N  = int(N)
    g  = int(g)
    m  = int(m)
    n  = int(n)
    if weights is None:
        weights = (m, n, m)

    if N <= 2:
        return (None, None, None)

    B = N + 2 * g - 2

    # r is a model degree. N is enough for the supported models because any
    # useful C1 must have actual pole degree < N.
    dims, poles = _precompute_model_data(m, n, weights, N)

    best_score = None
    best_r1    = None
    best_r2    = None
    best_dq    = None

    for r2 in range(0, N + 1):
        R2 = poles[r2]
        if R2 < 2 * g - 1:
            continue
        for r1 in range(r2 + 1, N + 1):
            R1 = poles[r1]
            if R1 >= N or R1 + R2 > B:
                continue
            k1 = dims[r1]
            k2 = dims[r2]
            k_q = k1 - k2
            if k_q <= 0:
                continue
            d1 = max(1, N - R1)
            d2d = max(1, R2 - 2 * g + 2)
            d_q = min(d1, d2d)

            if strategy == 'product':
                score = (k_q * d_q, d_q, k_q)
            elif strategy == 'balanced':
                score = (d_q, k_q)
            elif strategy == 'max_k_q':
                score = (k_q, d_q)
            else:
                score = (k_q * d_q, d_q, k_q)

            if best_score is None or score > best_score:
                best_score = score
                best_r1 = r1
                best_r2 = r2
                best_dq = d_q

    if best_r1 is not None:
        return (best_r1, best_r2, best_dq)

    return (None, None, None)


# ═══════════════════════════════════════════════════════════════════════
#  6.  BUILD CSS QUANTUM CODE
# ═══════════════════════════════════════════════════════════════════════

def build_css_quantum_code(F, f_poly, m, r1, r2, pts, g, weights=None,
                            compute_exact_d=False):
    r"""
    Build a CSS quantum code [[N, k_q, d_q]]_q from nested AG codes.
    """
    n = ZZ(f_poly.degree())
    N = len(pts)
    result = {
        'css_ok': False, 'N': N, 'k_q': 0, 'd_q': 0, 'd_q_bound': 0,
        'C1_params': (N, 0, 0), 'C2_params': (N, 0, 0),
        'skip_reason': None,
    }

    if N == 0:
        result['skip_reason'] = 'N=0'
        return result

    if weights is None:
        weights = (m, n, m)

    C1 = build_weighted_ag_code(F, f_poly, m, r1, pts, weights=weights,
                                 compute_exact_d=compute_exact_d)
    C2 = build_weighted_ag_code(F, f_poly, m, r2, pts, weights=weights,
                                 compute_exact_d=compute_exact_d)

    R1 = C1['pole_degree']
    R2 = C2['pole_degree']

    css_bound = N + 2 * g - 2
    if R1 + R2 > css_bound:
        result['skip_reason'] = (
            f'CSS_violated: pole1+pole2={R1+R2}>{css_bound}=N+2g-2'
        )
        return result

    result['C1_params'] = (C1['N'], C1['k'], C1['d'])

    result['C2_params'] = (C2['N'], C2['k'], C2['d'])

    k_q = C1['k'] - C2['k']
    if k_q <= 0:
        result['skip_reason'] = f'k_q={k_q}<=0'
        return result

    result['k_q'] = k_q

    d1_bound = max(1, N - R1)
    d2_dual_bound = max(1, R2 - 2 * g + 2)
    d_q_bound = min(d1_bound, d2_dual_bound)
    result['d_q_bound'] = d_q_bound

    d_q = d_q_bound
    is_classical = tuple(ZZ(w) for w in weights) == (1, 1, 1)
    if is_classical and C1['lc'] is not None and C2['lc'] is not None:
        try:
            if not C2['lc'].dual_code().is_subcode(C1['lc']):
                result['skip_reason'] = 'CSS_subcode_check_failed'
                return result
        except Exception:
            result['skip_reason'] = 'CSS_subcode_check_failed'
            return result

    if compute_exact_d and C2['lc'] is not None:
        try:
            C2_dual = C2['lc'].dual_code()
            if C1['lc'] is not None:
                css_verified = C2_dual.is_subcode(C1['lc'])
                if not css_verified:
                    result['skip_reason'] = 'CSS_subcode_check_failed'
                    return result
            d2_dual_exact = C2_dual.minimum_distance()
            d_q = min(C1['d'], d2_dual_exact)
        except Exception:
            d_q = d_q_bound

    result['d_q'] = d_q
    result['css_ok'] = True
    return result


# ═══════════════════════════════════════════════════════════════════════
#  7.  AUTOMORPHISM INFORMATION
# ═══════════════════════════════════════════════════════════════════════

def automorphism_lower_bound(F, m, n):
    q = F.cardinality()
    m = ZZ(m)
    has_zeta = (q - 1) % m == 0
    if has_zeta:
        return {
            'aut_lower_bound': m,
            'cyclic_order': m,
            'has_zeta_m': True,
            'aut_description': f'C_{m} (cyclic, y -> zeta_{m}*y)',
        }
    else:
        return {
            'aut_lower_bound': 1,
            'cyclic_order': 1,
            'has_zeta_m': False,
            'aut_description': f'trivial over F_{q} (zeta_{m} not in F_{q})',
        }


# ═══════════════════════════════════════════════════════════════════════
#  8.  SINGLE CURVE RUNNER
# ═══════════════════════════════════════════════════════════════════════

def run_one_m5_curve(q, coeffs, m=5, weights=None,
                      galois_label='', compute_exact_d=False,
                      css_strategy='product', verbose=True):
    t0 = time.time()

    row = {
        'q': q, 'm': m, 'coeffs': str(coeffs),
        'galois_label': galois_label,
        'weights': '', 'model_label': '', 'f_x': '',
        'genus': '', 'N': '', 'r1': '', 'r2': '',
        'd1_bound': '', 'd2_dual_bound': '', 'd_q_bound': '',
        'css_strategy': css_strategy,
        'C1': '', 'C2': '', 'Q': '',
        'aut_lower_bound': '', 'aut_description': '',
        'css_ok': False, 'skip_reason': '', 'time_s': '',
    }

    F = GF(q)
    Rx = PolynomialRing(F, 'x')
    f_coeffs_F = [F(c) for c in coeffs]
    f_poly = Rx(f_coeffs_F)
    row['f_x'] = str(f_poly)

    n_actual = f_poly.degree()
    if n_actual < 2:
        row['skip_reason'] = f'deg(f)={n_actual}<2_over_F_{q}'
        row['time_s'] = f'{time.time()-t0:.3f}'
        return row

    if weights is None:
        weights_use = (ZZ(m), ZZ(n_actual), ZZ(m))
    else:
        weights_use = tuple(ZZ(w) for w in weights)

    val = validate_superelliptic_curve(F, f_poly, m, weights_use)
    row['weights'] = str(val['weights']) if val['weights'] else ''
    row['model_label'] = val['model_label'] or ''
    row['genus'] = val['genus'] if val['genus'] is not None else ''

    if not val['ok']:
        if verbose:
            print(f"\n{'='*68}")
            print(f"Curve: y^{m} = {f_poly}   over GF({q})")
            print(f"Model: {val['model_label']}   weights={val['weights']}")
            print(f"SKIPPED: {val['skip_reason']}")
            print(f"{'='*68}")
        row['skip_reason'] = val['skip_reason']
        row['time_s'] = f'{time.time()-t0:.3f}'
        return row

    g = val['genus']
    n = val['n']

    if verbose:
        print(f"\n{'='*68}")
        print(f"Curve: y^{m} = {f_poly}   over GF({q})")
        print(f"Model: {val['model_label']}   weights={val['weights']}")
        print(f"Genus: g = {g},  Semigroup: <{m},{n}>")

    pts = affine_points_superelliptic(F, f_poly, m)
    N = len(pts)
    row['N'] = N

    if verbose:
        print(f"Affine rational points: N = {N}")

    if N == 0:
        row['skip_reason'] = 'N=0'
        row['time_s'] = f'{time.time()-t0:.3f}'
        return row

    # --- choose CSS parameters for this model ---
    css_result = choose_css_parameters(N, g, m, n, weights=weights_use,
                                       strategy=css_strategy)
    r1, r2, d_q_est = css_result

    if r1 is None:
        if verbose:
            print("Quantum: SKIPPED (no_valid_css_pair)")
            print(f"{'='*68}")
        row['skip_reason'] = 'no_valid_css_pair'
        row['time_s'] = f'{time.time()-t0:.3f}'
        return row

    row['r1'] = r1
    row['r2'] = r2
    basis1_pred = rr_basis_superelliptic(m, n, r1, weights=weights_use)
    basis2_pred = rr_basis_superelliptic(m, n, r2, weights=weights_use)
    R1_pred = pole_degree_of_basis(m, n, basis1_pred)
    R2_pred = pole_degree_of_basis(m, n, basis2_pred)

    row['d1_bound'] = max(1, N - R1_pred)
    row['d2_dual_bound'] = max(1, R2_pred - 2 * g + 2)
    row['d_q_bound'] = min(row['d1_bound'], row['d2_dual_bound'])

    if verbose:
        k1_pred = len(basis1_pred)
        k2_pred = len(basis2_pred)
        print(f"CSS candidate: r1={r1}, r2={r2}  (strategy={css_strategy})")
        print(f"  pole_degree(r1)={R1_pred}, pole_degree(r2)={R2_pred}")
        print(f"  l(r1)={k1_pred}, l(r2)={k2_pred}, k_Q={k1_pred-k2_pred}")
        print(f"  d(C1)>={row['d1_bound']}, d(C2^perp)>={row['d2_dual_bound']}")
        print(f"  candidate d_Q >= {row['d_q_bound']} (requires CSS nesting)")

    C1 = build_weighted_ag_code(F, f_poly, m, r1, pts, weights=weights_use,
                                 compute_exact_d=compute_exact_d)
    C2 = build_weighted_ag_code(F, f_poly, m, r2, pts, weights=weights_use,
                                 compute_exact_d=compute_exact_d)

    row['C1'] = f"[{C1['N']},{C1['k']},{C1['d']}]"
    row['C2'] = f"[{C2['N']},{C2['k']},{C2['d']}]"

    if verbose:
        print(f"C_1 = [{C1['N']},{C1['k']},{C1['d']}]")
        print(f"C_2 = [{C2['N']},{C2['k']},{C2['d']}]")

    Q = build_css_quantum_code(F, f_poly, m, r1, r2, pts, g,
                                weights=weights_use,
                                compute_exact_d=compute_exact_d)

    if Q['css_ok']:
        row['Q'] = f"[[{Q['N']},{Q['k_q']},{Q['d_q']}]]"
        row['d_q_bound'] = Q['d_q_bound']
        row['css_ok'] = True
    else:
        row['Q'] = ''
        row['skip_reason'] = Q.get('skip_reason', 'css_failed')

    if verbose:
        if Q['css_ok']:
            print(f"Quantum: [[{Q['N']},{Q['k_q']},{Q['d_q']}]]_{q}")
        else:
            print(f"Quantum: SKIPPED ({Q.get('skip_reason','')})")

    aut = automorphism_lower_bound(F, m, n)
    row['aut_lower_bound'] = aut['aut_lower_bound']
    row['aut_description'] = aut['aut_description']

    if verbose:
        print(f"Aut(X) >= {aut['aut_lower_bound']}")
        print(f"{'='*68}")

    row['time_s'] = f'{time.time()-t0:.3f}'
    return row


# ═══════════════════════════════════════════════════════════════════════
#  9.  DATABASE PROCESSOR
# ═══════════════════════════════════════════════════════════════════════

def _parse_aims7_columns(df_row, col_names):
    coeffs_raw = None
    for cname in ['coeffs',
                  '=HYPERLINK("https://www.lmfdb.org/knowledge/show/nf.defining_polynomial", "coeffs")']:
        if cname in col_names:
            coeffs_raw = df_row.get(cname, None)
            break
    if coeffs_raw is None:
        keys = list(df_row.keys())
        if len(keys) >= 2:
            coeffs_raw = df_row[keys[1]]

    gal_label = None
    for cname in ['galois_label',
                  '=HYPERLINK("https://www.lmfdb.org/knowledge/show/nf.galois_group", "galois_label")']:
        if cname in col_names:
            gal_label = df_row.get(cname, '')
            break
    if gal_label is None:
        keys = list(df_row.keys())
        gal_label = df_row[keys[3]] if len(keys) >= 4 else ''

    label = None
    for cname in ['label',
                  '=HYPERLINK("https://www.lmfdb.org/knowledge/show/nf.label", "label")']:
        if cname in col_names:
            label = df_row.get(cname, '')
            break
    if label is None:
        keys = list(df_row.keys())
        label = df_row.get(keys[0], '') if keys else ''

    math_gal = df_row.get('math_galois_notation', '')
    return coeffs_raw, gal_label, label, math_gal


def process_m5_database(csv_path, output_csv, q=11, m=5,
                         weights=None, limit=None,
                         compute_exact_d=False,
                         css_strategy='product',
                         verbose_every=50000):
    r"""
    Process the AIMS-7 database. Writes results INCREMENTALLY
    to output_csv so partial results survive crashes.
    """
    print(f"\n{'#'*68}")
    print(f"# m={m} Superelliptic Pipeline (FIXED balanced)")
    print(f"# Field: GF({q})")
    print(f"# Weights: {weights if weights else f'natural P({m},7,{m})'}")
    print(f"# CSS strategy: {css_strategy}")
    print(f"# Input:  {csv_path}")
    print(f"# Output: {output_csv}")
    print(f"{'#'*68}\n")

    rows_in = []
    with open(csv_path, 'r', encoding='utf-8') as fh:
        reader = _csv.DictReader(fh)
        col_names = reader.fieldnames
        for row in reader:
            rows_in.append(row)

    total = len(rows_in)
    print(f"Loaded {total} rows from database")
    if limit:
        rows_in = rows_in[:limit]
        print(f"Processing first {limit}")

    out_fields = [
        'label', 'galois_label', 'math_galois', 'coeffs',
        'q', 'm', 'weights', 'model_label', 'f_x',
        'genus', 'N', 'r1', 'r2',
        'd1_bound', 'd2_dual_bound', 'd_q_bound', 'css_strategy',
        'C1', 'C2', 'Q',
        'aut_lower_bound', 'aut_description',
        'css_ok', 'skip_reason', 'time_s',
    ]

    stats = {
        'total': len(rows_in), 'valid': 0,
        'ag_ok': 0, 'quantum_ok': 0, 'skipped': {},
    }

    t_start = time.time()

    # --- INCREMENTAL WRITE: open file once, flush regularly ---
    fh_out = open(output_csv, 'w', newline='', encoding='utf-8')
    writer = _csv.DictWriter(fh_out, fieldnames=out_fields,
                              extrasaction='ignore')
    writer.writeheader()

    for idx, raw_row in enumerate(rows_in):
        coeffs_raw, gal_label, label, math_gal = \
            _parse_aims7_columns(raw_row, col_names)

        try:
            coeffs = list(ast.literal_eval(str(coeffs_raw).strip()))
        except Exception:
            stats['skipped']['parse_error'] = \
                stats['skipped'].get('parse_error', 0) + 1
            continue

        row_result = run_one_m5_curve(
            q=q, coeffs=coeffs, m=m, weights=weights,
            galois_label=gal_label,
            compute_exact_d=compute_exact_d,
            css_strategy=css_strategy,
            verbose=False,
        )

        row_result['label'] = label
        row_result['math_galois'] = math_gal

        # Write immediately
        writer.writerow(row_result)

        if row_result.get('skip_reason', ''):
            reason = row_result['skip_reason']
            stats['skipped'][reason] = stats['skipped'].get(reason, 0) + 1
        else:
            stats['valid'] += 1
            if row_result.get('C1', ''):
                stats['ag_ok'] += 1
            if row_result.get('css_ok', False):
                stats['quantum_ok'] += 1

        if (idx + 1) % verbose_every == 0:
            fh_out.flush()
            elapsed = time.time() - t_start
            rate = (idx + 1) / elapsed if elapsed > 0 else 0
            eta = (len(rows_in) - idx - 1) / rate if rate > 0 else 0
            print(f"  [{idx+1}/{len(rows_in)}]  "
                  f"AG={stats['ag_ok']}  Q={stats['quantum_ok']}  "
                  f"skip={sum(stats['skipped'].values())}  "
                  f"({rate:.1f} rows/s, ETA {eta/60:.0f}min)")

    fh_out.close()

    elapsed_total = time.time() - t_start
    print(f"\n{'='*68}")
    print(f"DONE in {elapsed_total:.1f}s ({elapsed_total/60:.1f} min)")
    print(f"  Total rows:    {stats['total']}")
    print(f"  Valid curves:  {stats['valid']}")
    print(f"  AG codes:      {stats['ag_ok']}")
    print(f"  Quantum codes: {stats['quantum_ok']}")
    print(f"  Skipped:       {sum(stats['skipped'].values())}")
    for reason, count in sorted(stats['skipped'].items(),
                                 key=lambda x: -x[1]):
        print(f"    {reason}: {count}")
    print(f"  Output -> {output_csv}")
    print(f"{'='*68}")

    return stats


# ═══════════════════════════════════════════════════════════════════════
#  10.  EXAMPLE / SELF-TEST
# ═══════════════════════════════════════════════════════════════════════

def run_example():
    print("\n" + "="*68)
    print("  SELF-TEST: balanced CSS on y^5 = f(x)")
    print("="*68)

    test_polys = [
        ([1, 0, 1, -1, -1, 0, 0, 1], '7T7', 'S7'),   # N=35 over F_11
        ([1, 1, -1, 0, 1, -1, -1, 1], '7T7', 'S7'),   # N=15 over F_11
    ]

    fields = [11, 31, 61]
    weight_choices = [(5,7,5), (1,1,1)]

    for coeffs, gal, _ in test_polys:
        for q in fields:
            for w in weight_choices:
                row = run_one_m5_curve(
                    q=q, coeffs=coeffs, m=5, weights=w,
                    galois_label=gal,
                    css_strategy='product',
                    verbose=True,
                )

    print("\n" + "="*68)
    print("  Self-test complete.")
    print("="*68)


def _running_pipeline_file_directly():
    r"""
    Sage's load(...) executes files with __name__ == '__main__'.  Use argv[0]
    to distinguish direct self-test runs from database runner loads.
    """
    top_script = os.path.basename(str(sys.argv[0])).replace('.sage.py', '.sage')
    return top_script == 'm5_superelliptic_pipeline_FIXED.sage'


if __name__ == '__main__' and _running_pipeline_file_directly():
    run_example()


In [ ]:
#!/usr/bin/env sage
r"""
run_q31.sage — Run the fixed m=5 pipeline on GF(31) only.

Usage:
    sage run_q31.sage

Produces:
    m5_results_FIXED/m5_q31_P575.csv
    m5_results_FIXED/m5_q31_P111.csv
"""

import os, sys, time

def find_base_dir():
    candidates = []

    argv_dir = os.path.dirname(os.path.abspath(str(sys.argv[0])))
    if argv_dir.endswith('.sage.py'):
        argv_dir = os.path.dirname(argv_dir)
    candidates.append(argv_dir)

    # Notebook/Sage kernel fallback.  In notebooks, sys.argv[0] points inside
    # Sage's package directory, not to this project folder.
    candidates.append("/Users/jurimezini/Downloads/codex-ag-final")
    candidates.append(os.getcwd())

    for path in candidates:
        pipeline = os.path.join(path, "m5_superelliptic_pipeline_FIXED.sage")
        if os.path.exists(pipeline):
            return path

    raise RuntimeError("Could not find m5_superelliptic_pipeline_FIXED.sage")

BASE_DIR = find_base_dir()

load(os.path.join(BASE_DIR, "m5_superelliptic_pipeline_FIXED.sage"))

CSV_PATH   = "/Users/jurimezini/Library/CloudStorage/Dropbox/Sage_Galois7/AIMS-7.csv"
OUTPUT_DIR = os.path.join(BASE_DIR, "m5_results_FIXED")
os.makedirs(OUTPUT_DIR, exist_ok=True)

Q             = 31
CSS_STRATEGY  = "product"
TEST_LIMIT    = None
VERBOSE_EVERY = 50000

print("\n" + "=" * 70)
print(f"  FIELD: GF({Q})  —  two weight models")
print("=" * 70)

t0 = time.time()

out_575 = os.path.join(OUTPUT_DIR, f"m5_q{Q}_P575.csv")
print(f"\n>>> Starting P(5,7,5) -> {out_575}")
stats_575 = process_m5_database(
    csv_path=CSV_PATH, output_csv=out_575, q=Q, m=5,
    weights=(5,7,5), limit=TEST_LIMIT,
    compute_exact_d=False, css_strategy=CSS_STRATEGY,
    verbose_every=VERBOSE_EVERY,
)

out_111 = os.path.join(OUTPUT_DIR, f"m5_q{Q}_P111.csv")
print(f"\n>>> Starting P(1,1,1) -> {out_111}")
stats_111 = process_m5_database(
    csv_path=CSV_PATH, output_csv=out_111, q=Q, m=5,
    weights=(1,1,1), limit=TEST_LIMIT,
    compute_exact_d=False, css_strategy=CSS_STRATEGY,
    verbose_every=VERBOSE_EVERY,
)

elapsed = time.time() - t0
print(f"\n{'='*70}")
print(f"  GF({Q}) COMPLETE in {elapsed/60:.1f} minutes")
print(f"  P(5,7,5): {out_575}")
print(f"  P(1,1,1): {out_111}")
print(f"{'='*70}")


In [ ]:
#!/usr/bin/env sage
r"""
codex_classic_superelliptic.sage
================================
Codex Classic Codes for superelliptic curves  X: y^5 = f(x)
over finite fields GF(q).

MODEL NOTE:
  "Codex Classic" means the standard one-point AG-code construction on the
  normalized superelliptic function field F_q(x,y), y^m=f(x).  The basis is
  filtered by the true pole order at infinity:

      pole_order(x^i y^j) = i*m + j*deg(f).

  For m=5 and deg(f)=7 this is the semigroup <5,7>.  These are classical AG
  codes C_L(D,rP_infinity), and the CSS quantum codes are built from nested
  pairs of these AG codes.

KEY FIX (vs. original):
  The original choose_css_parameters used r1 = N-1, forcing d(C1) >= 1
  and hence d_Q = 1 for every code.  This version searches the SATURATED
  CSS line  r1 + r2 = N + 2g - 2  to find the pair that maximises the
  product  k_Q * d_Q_bound, giving nontrivial designed quantum distance.

PERFORMANCE FIX:
  The search is O(N) per curve (linear scan of the saturated line with
  precomputed Riemann-Roch dimensions), not O(N^2) as in the brute-force.

For m=5, deg(f)=7: genus g = 12, semigroup <5,7>.
"""

from sage.all import *
from sage.coding.linear_code import LinearCode
import csv as _csv
import ast
import os
import sys
import time


# ═══════════════════════════════════════════════════════════════════════
#  1.  VALIDATION
# ═══════════════════════════════════════════════════════════════════════

def validate_superelliptic_curve(F, f_poly, m, weights=None):
    r"""
    Validate that  y^m = f(x)  defines a smooth superelliptic curve
    over the finite field F, and that the requested weighted projective
    model is compatible with the weighted homogenization.
    """
    result = {
        'ok': False, 'skip_reason': None,
        'n': None, 'genus': None, 'weights': None, 'model_label': None,
    }

    m = ZZ(m)
    n = ZZ(f_poly.degree())
    result['n'] = n

    if n < 2:
        result['skip_reason'] = f'deg(f)={n}<2'
        return result

    g_test = gcd(m, n)
    if g_test != 1:
        result['skip_reason'] = f'gcd({m},{n})={g_test}!=1'
        return result

    p = F.characteristic()
    if p != 0 and (m * n) % p == 0:
        result['skip_reason'] = f'char={p}|m*n={m*n}'
        return result

    if f_poly.gcd(f_poly.derivative()) != 1:
        result['skip_reason'] = 'f_not_squarefree'
        return result

    genus = ZZ((m - 1) * (n - 1)) // 2
    result['genus'] = genus

    if weights is None:
        weights = (m, n, m)

    w0, w1, w2 = [ZZ(w) for w in weights]
    result['weights'] = (w0, w1, w2)
    result['model_label'] = f'P({w0},{w1},{w2})'

    is_classical = (w0 == 1 and w1 == 1 and w2 == 1)
    if not is_classical:
        target_degree = m * w1
        if target_degree != n * w0:
            result['skip_reason'] = (
                f'weights_inadmissible: y^{m} has weighted degree {target_degree}, '
                f'but x^{n} has weighted degree {n*w0}'
            )
            return result

        for exp, coeff in f_poly.dict().items():
            if coeff == 0:
                continue
            i = exp[0] if isinstance(exp, tuple) else exp
            z_degree = target_degree - i * w0
            if z_degree < 0 or z_degree % w2 != 0:
                result['skip_reason'] = (
                    f'weights_inadmissible: term x^{i} cannot be homogenized '
                    f'to weighted degree {target_degree} with z-weight {w2}'
                )
                return result

    result['ok'] = True
    return result


# ═══════════════════════════════════════════════════════════════════════
#  2.  AFFINE RATIONAL POINTS
# ═══════════════════════════════════════════════════════════════════════

def affine_points_superelliptic(F, f_poly, m):
    r"""
    Enumerate all affine F_q-rational points (a, b) with b^m = f(a).
    """
    m = ZZ(m)
    pts = []
    for a in F:
        fa = f_poly(a)
        for b in F:
            if b ** m == fa:
                pts.append((a, b))
    return pts


# ═══════════════════════════════════════════════════════════════════════
#  3.  RIEMANN-ROCH BASIS
# ═══════════════════════════════════════════════════════════════════════

def rr_basis_superelliptic(m, n, r, weights=None):
    r"""
    Monomial basis for the selected one-point model.

    Codex Classic / one-point AG model:
        i*m + j*n <= r

    Distance bounds still use the actual pole order i*m+j*n.
    """
    m, n, r = ZZ(m), ZZ(n), ZZ(r)
    if weights is None:
        weights = (m, n, m)
    w0, w1, _ = [ZZ(w) for w in weights]

    basis = []
    for j in range(m):
        if j * w1 > r:
            continue
        max_i = (r - j * w1) // w0
        for i in range(int(max_i) + 1):
            basis.append((i, j))
    basis.sort(key=lambda ij: ij[0] * m + ij[1] * n)
    return basis


def pole_degree_of_basis(m, n, basis):
    """Maximum pole order at P_inf among basis monomials."""
    if not basis:
        return -1
    return max(ZZ(i) * ZZ(m) + ZZ(j) * ZZ(n) for i, j in basis)


def rr_dim(m, n, r, weights=None):
    """Dimension of the selected model basis by lattice-point counting."""
    return len(rr_basis_superelliptic(m, n, r, weights=weights))


def _precompute_model_data(m, n, weights, max_r):
    """Precompute dimensions and actual pole degrees for model degrees."""
    dims = []
    poles = []
    for r in range(max_r + 1):
        basis = rr_basis_superelliptic(m, n, r, weights=weights)
        dims.append(len(basis))
        poles.append(pole_degree_of_basis(m, n, basis))
    return dims, poles


# ═══════════════════════════════════════════════════════════════════════
#  4.  BUILD AG CODE
# ═══════════════════════════════════════════════════════════════════════

def build_weighted_ag_code(F, f_poly, m, r, pts, weights=None,
                           compute_exact_d=False):
    r"""
    Construct the evaluation AG code  C_L(D, r*P_inf).
    """
    m = ZZ(m)
    n = ZZ(f_poly.degree())
    N = len(pts)

    if N == 0:
        return {'N': 0, 'k': 0, 'd': 0, 'd_goppa': 0,
                'G_mat': None, 'basis': [], 'lc': None}

    if weights is None:
        weights = (m, n, m)

    basis = rr_basis_superelliptic(m, n, r, weights=weights)
    pole_degree = pole_degree_of_basis(m, n, basis)

    rows = []
    for (i, j) in basis:
        row = [F(a) ** i * F(b) ** j for (a, b) in pts]
        rows.append(row)

    if not rows:
        return {'N': N, 'k': 0, 'd': 0, 'd_goppa': max(0, N - pole_degree),
                'pole_degree': pole_degree, 'G_mat': matrix(F, 0, N),
                'basis': basis, 'lc': None}

    G_mat = matrix(F, rows)
    k = G_mat.rank()
    d_goppa = max(0, N - pole_degree)

    lc = None
    d = d_goppa

    if k > 0:
        pivots = G_mat.pivot_rows()
        G_red = G_mat.matrix_from_rows(pivots)
        try:
            lc = LinearCode(G_red)
            if compute_exact_d and k < 20 and N < 80:
                d = lc.minimum_distance()
            else:
                d = d_goppa
        except Exception:
            d = d_goppa

    return {
        'N': N, 'k': k, 'd': d, 'd_goppa': d_goppa,
        'pole_degree': pole_degree,
        'G_mat': G_mat, 'basis': basis, 'lc': lc,
    }


# ═══════════════════════════════════════════════════════════════════════
#  5.  CSS PARAMETER SELECTION  (FIXED — O(N) balanced search)
# ═══════════════════════════════════════════════════════════════════════

def choose_css_parameters(N, g, m, n, weights=None, strategy='product'):
    r"""
    Choose model degrees r1, r2 for the CSS construction.

    The basis is selected by model degree, but CSS validity and distance
    bounds are checked using actual pole degrees R1 and R2:
        R1 + R2 <= N + 2g - 2,
        d(C1) >= N - R1,
        d(C2^perp) >= R2 - (2g - 2).

    Strategies (all search the saturated line):
      'product'   -- maximise  k_Q * d_q_bound  (recommended)
      'balanced'  -- maximise  d_q_bound  first, then k_Q
                     (gives highest certified distance, but k_Q ~ 1)
      'max_k_q'   -- maximise  k_Q  first, then d_q_bound
                     (highest dimension, but d_q_bound ~ 1)

    OUTPUT:
      (r1, r2, d_q_bound)  or  (None, None, None)
    """
    N  = int(N)
    g  = int(g)
    m  = int(m)
    n  = int(n)
    if weights is None:
        weights = (m, n, m)

    if N <= 2:
        return (None, None, None)

    B = N + 2 * g - 2

    # r is a model degree. N is enough for the supported models because any
    # useful C1 must have actual pole degree < N.
    dims, poles = _precompute_model_data(m, n, weights, N)

    best_score = None
    best_r1    = None
    best_r2    = None
    best_dq    = None

    for r2 in range(0, N + 1):
        R2 = poles[r2]
        if R2 < 2 * g - 1:
            continue
        for r1 in range(r2 + 1, N + 1):
            R1 = poles[r1]
            if R1 >= N or R1 + R2 > B:
                continue
            k1 = dims[r1]
            k2 = dims[r2]
            k_q = k1 - k2
            if k_q <= 0:
                continue
            d1 = max(1, N - R1)
            d2d = max(1, R2 - 2 * g + 2)
            d_q = min(d1, d2d)

            if strategy == 'product':
                score = (k_q * d_q, d_q, k_q)
            elif strategy == 'balanced':
                score = (d_q, k_q)
            elif strategy == 'max_k_q':
                score = (k_q, d_q)
            else:
                score = (k_q * d_q, d_q, k_q)

            if best_score is None or score > best_score:
                best_score = score
                best_r1 = r1
                best_r2 = r2
                best_dq = d_q

    if best_r1 is not None:
        return (best_r1, best_r2, best_dq)

    return (None, None, None)


# ═══════════════════════════════════════════════════════════════════════
#  6.  BUILD CSS QUANTUM CODE
# ═══════════════════════════════════════════════════════════════════════

def build_css_quantum_code(F, f_poly, m, r1, r2, pts, g, weights=None,
                            compute_exact_d=False):
    r"""
    Build a CSS quantum code [[N, k_q, d_q]]_q from nested AG codes.
    """
    n = ZZ(f_poly.degree())
    N = len(pts)
    result = {
        'css_ok': False, 'N': N, 'k_q': 0, 'd_q': 0, 'd_q_bound': 0,
        'C1_params': (N, 0, 0), 'C2_params': (N, 0, 0),
        'skip_reason': None,
    }

    if N == 0:
        result['skip_reason'] = 'N=0'
        return result

    if weights is None:
        weights = (m, n, m)

    C1 = build_weighted_ag_code(F, f_poly, m, r1, pts, weights=weights,
                                 compute_exact_d=compute_exact_d)
    C2 = build_weighted_ag_code(F, f_poly, m, r2, pts, weights=weights,
                                 compute_exact_d=compute_exact_d)

    R1 = C1['pole_degree']
    R2 = C2['pole_degree']

    css_bound = N + 2 * g - 2
    if R1 + R2 > css_bound:
        result['skip_reason'] = (
            f'CSS_violated: pole1+pole2={R1+R2}>{css_bound}=N+2g-2'
        )
        return result

    result['C1_params'] = (C1['N'], C1['k'], C1['d'])

    result['C2_params'] = (C2['N'], C2['k'], C2['d'])

    k_q = C1['k'] - C2['k']
    if k_q <= 0:
        result['skip_reason'] = f'k_q={k_q}<=0'
        return result

    result['k_q'] = k_q

    d1_bound = max(1, N - R1)
    d2_dual_bound = max(1, R2 - 2 * g + 2)
    d_q_bound = min(d1_bound, d2_dual_bound)
    result['d_q_bound'] = d_q_bound

    d_q = d_q_bound
    is_classical = tuple(ZZ(w) for w in weights) == (1, 1, 1)
    if is_classical and C1['lc'] is not None and C2['lc'] is not None:
        try:
            if not C2['lc'].dual_code().is_subcode(C1['lc']):
                result['skip_reason'] = 'CSS_subcode_check_failed'
                return result
        except Exception:
            result['skip_reason'] = 'CSS_subcode_check_failed'
            return result

    if compute_exact_d and C2['lc'] is not None:
        try:
            C2_dual = C2['lc'].dual_code()
            if C1['lc'] is not None:
                css_verified = C2_dual.is_subcode(C1['lc'])
                if not css_verified:
                    result['skip_reason'] = 'CSS_subcode_check_failed'
                    return result
            d2_dual_exact = C2_dual.minimum_distance()
            d_q = min(C1['d'], d2_dual_exact)
        except Exception:
            d_q = d_q_bound

    result['d_q'] = d_q
    result['css_ok'] = True
    return result


# ═══════════════════════════════════════════════════════════════════════
#  7.  AUTOMORPHISM INFORMATION
# ═══════════════════════════════════════════════════════════════════════

def automorphism_lower_bound(F, m, n):
    q = F.cardinality()
    m = ZZ(m)
    has_zeta = (q - 1) % m == 0
    if has_zeta:
        return {
            'aut_lower_bound': m,
            'cyclic_order': m,
            'has_zeta_m': True,
            'aut_description': f'C_{m} (cyclic, y -> zeta_{m}*y)',
        }
    else:
        return {
            'aut_lower_bound': 1,
            'cyclic_order': 1,
            'has_zeta_m': False,
            'aut_description': f'trivial over F_{q} (zeta_{m} not in F_{q})',
        }


def automorphism_group_affine_exact(F, f_poly, m, max_generators=12):
    r"""
    Exact finite-field automorphisms for y^m=f(x) fixing P_infinity.

    For gcd(m,deg(f))=1 there is a unique point at infinity.  Hence every
    curve automorphism fixes P_infinity, and x must map to u*x+v.  Since
    char(F) does not divide m, y maps to beta*y.  We therefore test:

        f(u*x+v) = beta^m * f(x).

    Returns all automorphisms over F of this form as triples (u,v,beta).
    """
    m = ZZ(m)
    Rx = f_poly.parent()
    x = Rx.gen()
    f_poly = Rx(f_poly)

    automorphisms = []
    reduced_affine = []

    beta_by_power = {}
    for beta in F:
        beta_by_power.setdefault(beta ** m, []).append(beta)

    for u in F:
        if u == 0:
            continue
        for v in F:
            transformed = f_poly(u * x + v)
            alpha = None
            if f_poly != 0:
                lc = f_poly.leading_coefficient()
                if lc != 0:
                    alpha = transformed.leading_coefficient() / lc
            if alpha is None:
                continue
            if transformed != alpha * f_poly:
                continue

            betas = beta_by_power.get(F(alpha), [])
            if not betas:
                continue

            reduced_affine.append((u, v, F(alpha)))
            for beta in betas:
                automorphisms.append((u, v, beta))

    non_identity = [
        aut for aut in automorphisms
        if not (aut[0] == F(1) and aut[1] == F(0) and aut[2] == F(1))
    ]
    sample_source = non_identity + [
        aut for aut in automorphisms
        if aut[0] == F(1) and aut[1] == F(0) and aut[2] == F(1)
    ]

    sample = []
    for u, v, beta in sample_source[:max_generators]:
        sample.append(f"x->{u}*x+{v}; y->{beta}*y")

    return {
        'aut_exact_order': len(automorphisms),
        'reduced_affine_order': len(reduced_affine),
        'deck_order': len(beta_by_power.get(F(1), [])),
        'automorphisms': automorphisms,
        'reduced_affine': reduced_affine,
        'sample_generators': sample,
        'description': (
            f"Aut_Fq order {len(automorphisms)}; "
            f"reduced affine order {len(reduced_affine)}; "
            f"deck order {len(beta_by_power.get(F(1), []))}"
        ),
    }


# ═══════════════════════════════════════════════════════════════════════
#  8.  SINGLE CURVE RUNNER
# ═══════════════════════════════════════════════════════════════════════

def run_one_m5_curve(q, coeffs, m=5, weights=None,
                      galois_label='', compute_exact_d=False,
                      css_strategy='product', compute_aut_exact=False,
                      verbose=True):
    t0 = time.time()

    row = {
        'q': q, 'm': m, 'coeffs': str(coeffs),
        'galois_label': galois_label,
        'weights': '', 'model_label': '', 'f_x': '',
        'genus': '', 'N': '', 'r1': '', 'r2': '',
        'd1_bound': '', 'd2_dual_bound': '', 'd_q_bound': '',
        'css_strategy': css_strategy,
        'C1': '', 'C2': '', 'Q': '',
        'aut_lower_bound': '', 'aut_description': '',
        'aut_exact_order': '', 'aut_reduced_order': '',
        'aut_exact_description': '', 'aut_sample': '',
        'css_ok': False, 'skip_reason': '', 'time_s': '',
    }

    F = GF(q)
    Rx = PolynomialRing(F, 'x')
    f_coeffs_F = [F(c) for c in coeffs]
    f_poly = Rx(f_coeffs_F)
    row['f_x'] = str(f_poly)

    n_actual = f_poly.degree()
    if n_actual < 2:
        row['skip_reason'] = f'deg(f)={n_actual}<2_over_F_{q}'
        row['time_s'] = f'{time.time()-t0:.3f}'
        return row

    if weights is None:
        weights_use = (ZZ(m), ZZ(n_actual), ZZ(m))
    else:
        weights_use = tuple(ZZ(w) for w in weights)

    val = validate_superelliptic_curve(F, f_poly, m, weights_use)
    row['weights'] = str(val['weights']) if val['weights'] else ''
    row['model_label'] = val['model_label'] or ''
    row['genus'] = val['genus'] if val['genus'] is not None else ''

    if not val['ok']:
        if verbose:
            print(f"\n{'='*68}")
            print(f"Curve: y^{m} = {f_poly}   over GF({q})")
            print(f"Model: {val['model_label']}   weights={val['weights']}")
            print(f"SKIPPED: {val['skip_reason']}")
            print(f"{'='*68}")
        row['skip_reason'] = val['skip_reason']
        row['time_s'] = f'{time.time()-t0:.3f}'
        return row

    g = val['genus']
    n = val['n']

    aut = automorphism_lower_bound(F, m, n)
    row['aut_lower_bound'] = aut['aut_lower_bound']
    row['aut_description'] = aut['aut_description']

    if compute_aut_exact:
        aut_exact = automorphism_group_affine_exact(F, f_poly, m)
        row['aut_exact_order'] = aut_exact['aut_exact_order']
        row['aut_reduced_order'] = aut_exact['reduced_affine_order']
        row['aut_exact_description'] = aut_exact['description']
        row['aut_sample'] = " | ".join(aut_exact['sample_generators'])

    if verbose:
        print(f"\n{'='*68}")
        print(f"Curve: y^{m} = {f_poly}   over GF({q})")
        print(f"Model: {val['model_label']}   weights={val['weights']}")
        print(f"Genus: g = {g},  Semigroup: <{m},{n}>")
        print(f"Aut(X) >= {aut['aut_lower_bound']}")
        if compute_aut_exact:
            print(f"Aut_Fq(X): order {row['aut_exact_order']}")
            print(f"  reduced affine order: {row['aut_reduced_order']}")
            if row['aut_sample']:
                print(f"  sample: {row['aut_sample'].split(' | ')[0]}")

    pts = affine_points_superelliptic(F, f_poly, m)
    N = len(pts)
    row['N'] = N

    if verbose:
        print(f"Affine rational points: N = {N}")

    if N == 0:
        row['skip_reason'] = 'N=0'
        row['time_s'] = f'{time.time()-t0:.3f}'
        return row

    # --- choose CSS parameters for this model ---
    css_result = choose_css_parameters(N, g, m, n, weights=weights_use,
                                       strategy=css_strategy)
    r1, r2, d_q_est = css_result

    if r1 is None:
        if verbose:
            print("Quantum: SKIPPED (no_valid_css_pair)")
            print(f"{'='*68}")
        row['skip_reason'] = 'no_valid_css_pair'
        row['time_s'] = f'{time.time()-t0:.3f}'
        return row

    row['r1'] = r1
    row['r2'] = r2
    basis1_pred = rr_basis_superelliptic(m, n, r1, weights=weights_use)
    basis2_pred = rr_basis_superelliptic(m, n, r2, weights=weights_use)
    R1_pred = pole_degree_of_basis(m, n, basis1_pred)
    R2_pred = pole_degree_of_basis(m, n, basis2_pred)

    row['d1_bound'] = max(1, N - R1_pred)
    row['d2_dual_bound'] = max(1, R2_pred - 2 * g + 2)
    row['d_q_bound'] = min(row['d1_bound'], row['d2_dual_bound'])

    if verbose:
        k1_pred = len(basis1_pred)
        k2_pred = len(basis2_pred)
        print(f"CSS candidate: r1={r1}, r2={r2}  (strategy={css_strategy})")
        print(f"  pole_degree(r1)={R1_pred}, pole_degree(r2)={R2_pred}")
        print(f"  l(r1)={k1_pred}, l(r2)={k2_pred}, k_Q={k1_pred-k2_pred}")
        print(f"  d(C1)>={row['d1_bound']}, d(C2^perp)>={row['d2_dual_bound']}")
        print(f"  candidate d_Q >= {row['d_q_bound']} (requires CSS nesting)")

    C1 = build_weighted_ag_code(F, f_poly, m, r1, pts, weights=weights_use,
                                 compute_exact_d=compute_exact_d)
    C2 = build_weighted_ag_code(F, f_poly, m, r2, pts, weights=weights_use,
                                 compute_exact_d=compute_exact_d)

    row['C1'] = f"[{C1['N']},{C1['k']},{C1['d']}]"
    row['C2'] = f"[{C2['N']},{C2['k']},{C2['d']}]"

    if verbose:
        print(f"C_1 = [{C1['N']},{C1['k']},{C1['d']}]")
        print(f"C_2 = [{C2['N']},{C2['k']},{C2['d']}]")

    Q = build_css_quantum_code(F, f_poly, m, r1, r2, pts, g,
                                weights=weights_use,
                                compute_exact_d=compute_exact_d)

    if Q['css_ok']:
        row['Q'] = f"[[{Q['N']},{Q['k_q']},{Q['d_q']}]]"
        row['d_q_bound'] = Q['d_q_bound']
        row['css_ok'] = True
    else:
        row['Q'] = ''
        row['skip_reason'] = Q.get('skip_reason', 'css_failed')

    if verbose:
        if Q['css_ok']:
            print(f"Quantum: [[{Q['N']},{Q['k_q']},{Q['d_q']}]]_{q}")
        else:
            print(f"Quantum: SKIPPED ({Q.get('skip_reason','')})")

    if verbose:
        print(f"{'='*68}")

    row['time_s'] = f'{time.time()-t0:.3f}'
    return row


# ═══════════════════════════════════════════════════════════════════════
#  9.  DATABASE PROCESSOR
# ═══════════════════════════════════════════════════════════════════════

def _parse_aims7_columns(df_row, col_names):
    coeffs_raw = None
    for cname in ['coeffs',
                  '=HYPERLINK("https://www.lmfdb.org/knowledge/show/nf.defining_polynomial", "coeffs")']:
        if cname in col_names:
            coeffs_raw = df_row.get(cname, None)
            break
    if coeffs_raw is None:
        keys = list(df_row.keys())
        if len(keys) >= 2:
            coeffs_raw = df_row[keys[1]]

    gal_label = None
    for cname in ['galois_label',
                  '=HYPERLINK("https://www.lmfdb.org/knowledge/show/nf.galois_group", "galois_label")']:
        if cname in col_names:
            gal_label = df_row.get(cname, '')
            break
    if gal_label is None:
        keys = list(df_row.keys())
        gal_label = df_row[keys[3]] if len(keys) >= 4 else ''

    label = None
    for cname in ['label',
                  '=HYPERLINK("https://www.lmfdb.org/knowledge/show/nf.label", "label")']:
        if cname in col_names:
            label = df_row.get(cname, '')
            break
    if label is None:
        keys = list(df_row.keys())
        label = df_row.get(keys[0], '') if keys else ''

    math_gal = df_row.get('math_galois_notation', '')
    return coeffs_raw, gal_label, label, math_gal


def process_m5_database(csv_path, output_csv, q=11, m=5,
                         weights=None, limit=None,
                         compute_exact_d=False,
                         compute_aut_exact=False,
                         css_strategy='product',
                         verbose_every=50000):
    r"""
    Process the AIMS-7 database. Writes results INCREMENTALLY
    to output_csv so partial results survive crashes.
    """
    print(f"\n{'#'*68}")
    print(f"# Codex Classic Codes: m={m} superelliptic AG/CSS pipeline")
    print(f"# Field: GF({q})")
    print(f"# Weights: {weights if weights else f'natural P({m},7,{m})'}")
    print(f"# CSS strategy: {css_strategy}")
    print(f"# Input:  {csv_path}")
    print(f"# Output: {output_csv}")
    print(f"{'#'*68}\n")

    rows_in = []
    with open(csv_path, 'r', encoding='utf-8') as fh:
        reader = _csv.DictReader(fh)
        col_names = reader.fieldnames
        for row in reader:
            rows_in.append(row)

    total = len(rows_in)
    print(f"Loaded {total} rows from database")
    if limit:
        rows_in = rows_in[:limit]
        print(f"Processing first {limit}")

    out_fields = [
        'label', 'galois_label', 'math_galois', 'coeffs',
        'q', 'm', 'weights', 'model_label', 'f_x',
        'genus', 'N', 'r1', 'r2',
        'd1_bound', 'd2_dual_bound', 'd_q_bound', 'css_strategy',
        'C1', 'C2', 'Q',
        'aut_lower_bound', 'aut_description',
        'aut_exact_order', 'aut_reduced_order',
        'aut_exact_description', 'aut_sample',
        'css_ok', 'skip_reason', 'time_s',
    ]

    stats = {
        'total': len(rows_in), 'valid': 0,
        'ag_ok': 0, 'quantum_ok': 0, 'skipped': {},
    }

    t_start = time.time()

    # --- INCREMENTAL WRITE: open file once, flush regularly ---
    fh_out = open(output_csv, 'w', newline='', encoding='utf-8')
    writer = _csv.DictWriter(fh_out, fieldnames=out_fields,
                              extrasaction='ignore')
    writer.writeheader()

    for idx, raw_row in enumerate(rows_in):
        coeffs_raw, gal_label, label, math_gal = \
            _parse_aims7_columns(raw_row, col_names)

        try:
            coeffs = list(ast.literal_eval(str(coeffs_raw).strip()))
        except Exception:
            stats['skipped']['parse_error'] = \
                stats['skipped'].get('parse_error', 0) + 1
            continue

        row_result = run_one_m5_curve(
            q=q, coeffs=coeffs, m=m, weights=weights,
            galois_label=gal_label,
            compute_exact_d=compute_exact_d,
            compute_aut_exact=compute_aut_exact,
            css_strategy=css_strategy,
            verbose=False,
        )

        row_result['label'] = label
        row_result['math_galois'] = math_gal

        # Write immediately
        writer.writerow(row_result)

        if row_result.get('C1', ''):
            stats['ag_ok'] += 1

        if row_result.get('css_ok', False):
            stats['quantum_ok'] += 1

        if row_result.get('skip_reason', ''):
            reason = row_result['skip_reason']
            stats['skipped'][reason] = stats['skipped'].get(reason, 0) + 1
        else:
            stats['valid'] += 1

        if (idx + 1) % verbose_every == 0:
            fh_out.flush()
            elapsed = time.time() - t_start
            rate = (idx + 1) / elapsed if elapsed > 0 else 0
            eta = (len(rows_in) - idx - 1) / rate if rate > 0 else 0
            print(f"  [{idx+1}/{len(rows_in)}]  "
                  f"AG={stats['ag_ok']}  Q={stats['quantum_ok']}  "
                  f"skip={sum(stats['skipped'].values())}  "
                  f"({rate:.1f} rows/s, ETA {eta/60:.0f}min)")

    fh_out.close()

    elapsed_total = time.time() - t_start
    print(f"\n{'='*68}")
    print(f"DONE in {elapsed_total:.1f}s ({elapsed_total/60:.1f} min)")
    print(f"  Total rows:    {stats['total']}")
    print(f"  Valid quantum rows: {stats['valid']}")
    print(f"  Classical AG rows:  {stats['ag_ok']}")
    print(f"  Quantum codes:      {stats['quantum_ok']}")
    print(f"  Skipped:       {sum(stats['skipped'].values())}")
    for reason, count in sorted(stats['skipped'].items(),
                                 key=lambda x: -x[1]):
        print(f"    {reason}: {count}")
    print(f"  Output -> {output_csv}")
    print(f"{'='*68}")

    return stats


# ═══════════════════════════════════════════════════════════════════════
#  10.  EXAMPLE / SELF-TEST
# ═══════════════════════════════════════════════════════════════════════

def run_example():
    print("\n" + "="*68)
    print("  SELF-TEST: balanced CSS on y^5 = f(x)")
    print("="*68)

    test_polys = [
        ([1, 0, 1, -1, -1, 0, 0, 1], '7T7', 'S7'),   # N=35 over F_11
        ([1, 1, -1, 0, 1, -1, -1, 1], '7T7', 'S7'),   # N=15 over F_11
    ]

    fields = [11, 31, 61]
    weight_choices = [(5,7,5)]

    for coeffs, gal, _ in test_polys:
        for q in fields:
            for w in weight_choices:
                row = run_one_m5_curve(
                    q=q, coeffs=coeffs, m=5, weights=w,
                    galois_label=gal,
                    css_strategy='product',
                    compute_aut_exact=True,
                    verbose=True,
                )

    print("\n" + "="*68)
    print("  Self-test complete.")
    print("="*68)


def _running_pipeline_file_directly():
    r"""
    Sage's load(...) executes files with __name__ == '__main__'.  Use argv[0]
    to distinguish direct self-test runs from database runner loads.
    """
    top_script = os.path.basename(str(sys.argv[0])).replace('.sage.py', '.sage')
    return top_script == 'codex_classic_superelliptic.sage'


if __name__ == '__main__' and _running_pipeline_file_directly():
    run_example()


In [ ]:
#!/usr/bin/env sage
r"""
codex_classic_superelliptic.sage
================================
Codex Classic Codes for superelliptic curves  X: y^5 = f(x)
over finite fields GF(q).

MODEL NOTE:
  "Codex Classic" means the standard one-point AG-code construction on the
  normalized superelliptic function field F_q(x,y), y^m=f(x).  The basis is
  filtered by the true pole order at infinity:

      pole_order(x^i y^j) = i*m + j*deg(f).

  For m=5 and deg(f)=7 this is the semigroup <5,7>.  These are classical AG
  codes C_L(D,rP_infinity), and the CSS quantum codes are built from nested
  pairs of these AG codes.

KEY FIX (vs. original):
  The original choose_css_parameters used r1 = N-1, forcing d(C1) >= 1
  and hence d_Q = 1 for every code.  This version searches the SATURATED
  CSS line  r1 + r2 = N + 2g - 2  to find the pair that maximises the
  product  k_Q * d_Q_bound, giving nontrivial designed quantum distance.

PERFORMANCE FIX:
  The search is O(N) per curve (linear scan of the saturated line with
  precomputed Riemann-Roch dimensions), not O(N^2) as in the brute-force.

For m=5, deg(f)=7: genus g = 12, semigroup <5,7>.
"""

from sage.all import *
from sage.coding.linear_code import LinearCode
import csv as _csv
import ast
import os
import sys
import time


# ═══════════════════════════════════════════════════════════════════════
#  1.  VALIDATION
# ═══════════════════════════════════════════════════════════════════════

def validate_superelliptic_curve(F, f_poly, m, weights=None):
    r"""
    Validate that  y^m = f(x)  defines a smooth superelliptic curve
    over the finite field F, and that the requested weighted projective
    model is compatible with the weighted homogenization.
    """
    result = {
        'ok': False, 'skip_reason': None,
        'n': None, 'genus': None, 'weights': None, 'model_label': None,
    }

    m = ZZ(m)
    n = ZZ(f_poly.degree())
    result['n'] = n

    if n < 2:
        result['skip_reason'] = f'deg(f)={n}<2'
        return result

    g_test = gcd(m, n)
    if g_test != 1:
        result['skip_reason'] = f'gcd({m},{n})={g_test}!=1'
        return result

    p = F.characteristic()
    if p != 0 and (m * n) % p == 0:
        result['skip_reason'] = f'char={p}|m*n={m*n}'
        return result

    if f_poly.gcd(f_poly.derivative()) != 1:
        result['skip_reason'] = 'f_not_squarefree'
        return result

    genus = ZZ((m - 1) * (n - 1)) // 2
    result['genus'] = genus

    if weights is None:
        weights = (m, n, m)

    w0, w1, w2 = [ZZ(w) for w in weights]
    result['weights'] = (w0, w1, w2)
    result['model_label'] = f'P({w0},{w1},{w2})'

    is_classical = (w0 == 1 and w1 == 1 and w2 == 1)
    if not is_classical:
        target_degree = m * w1
        if target_degree != n * w0:
            result['skip_reason'] = (
                f'weights_inadmissible: y^{m} has weighted degree {target_degree}, '
                f'but x^{n} has weighted degree {n*w0}'
            )
            return result

        for exp, coeff in f_poly.dict().items():
            if coeff == 0:
                continue
            i = exp[0] if isinstance(exp, tuple) else exp
            z_degree = target_degree - i * w0
            if z_degree < 0 or z_degree % w2 != 0:
                result['skip_reason'] = (
                    f'weights_inadmissible: term x^{i} cannot be homogenized '
                    f'to weighted degree {target_degree} with z-weight {w2}'
                )
                return result

    result['ok'] = True
    return result


# ═══════════════════════════════════════════════════════════════════════
#  2.  AFFINE RATIONAL POINTS
# ═══════════════════════════════════════════════════════════════════════

def affine_points_superelliptic(F, f_poly, m):
    r"""
    Enumerate all affine F_q-rational points (a, b) with b^m = f(a).
    """
    m = ZZ(m)
    pts = []
    for a in F:
        fa = f_poly(a)
        for b in F:
            if b ** m == fa:
                pts.append((a, b))
    return pts


# ═══════════════════════════════════════════════════════════════════════
#  3.  RIEMANN-ROCH BASIS
# ═══════════════════════════════════════════════════════════════════════

def rr_basis_superelliptic(m, n, r, weights=None):
    r"""
    Monomial basis for the selected one-point model.

    Codex Classic / one-point AG model:
        i*m + j*n <= r

    Distance bounds still use the actual pole order i*m+j*n.
    """
    m, n, r = ZZ(m), ZZ(n), ZZ(r)
    if weights is None:
        weights = (m, n, m)
    w0, w1, _ = [ZZ(w) for w in weights]

    basis = []
    for j in range(m):
        if j * w1 > r:
            continue
        max_i = (r - j * w1) // w0
        for i in range(int(max_i) + 1):
            basis.append((i, j))
    basis.sort(key=lambda ij: ij[0] * m + ij[1] * n)
    return basis


def pole_degree_of_basis(m, n, basis):
    """Maximum pole order at P_inf among basis monomials."""
    if not basis:
        return -1
    return max(ZZ(i) * ZZ(m) + ZZ(j) * ZZ(n) for i, j in basis)


def rr_dim(m, n, r, weights=None):
    """Dimension of the selected model basis by lattice-point counting."""
    return len(rr_basis_superelliptic(m, n, r, weights=weights))


def _precompute_model_data(m, n, weights, max_r):
    """Precompute dimensions and actual pole degrees for model degrees."""
    dims = []
    poles = []
    for r in range(max_r + 1):
        basis = rr_basis_superelliptic(m, n, r, weights=weights)
        dims.append(len(basis))
        poles.append(pole_degree_of_basis(m, n, basis))
    return dims, poles


# ═══════════════════════════════════════════════════════════════════════
#  4.  BUILD AG CODE
# ═══════════════════════════════════════════════════════════════════════

def build_weighted_ag_code(F, f_poly, m, r, pts, weights=None,
                           compute_exact_d=False):
    r"""
    Construct the evaluation AG code  C_L(D, r*P_inf).
    """
    m = ZZ(m)
    n = ZZ(f_poly.degree())
    N = len(pts)

    if N == 0:
        return {'N': 0, 'k': 0, 'd': 0, 'd_goppa': 0,
                'G_mat': None, 'basis': [], 'lc': None}

    if weights is None:
        weights = (m, n, m)

    basis = rr_basis_superelliptic(m, n, r, weights=weights)
    pole_degree = pole_degree_of_basis(m, n, basis)

    rows = []
    for (i, j) in basis:
        row = [F(a) ** i * F(b) ** j for (a, b) in pts]
        rows.append(row)

    if not rows:
        return {'N': N, 'k': 0, 'd': 0, 'd_goppa': max(0, N - pole_degree),
                'pole_degree': pole_degree, 'G_mat': matrix(F, 0, N),
                'basis': basis, 'lc': None}

    G_mat = matrix(F, rows)
    k = G_mat.rank()
    d_goppa = max(0, N - pole_degree)

    lc = None
    d = d_goppa

    if k > 0:
        pivots = G_mat.pivot_rows()
        G_red = G_mat.matrix_from_rows(pivots)
        try:
            lc = LinearCode(G_red)
            if compute_exact_d and k < 20 and N < 80:
                d = lc.minimum_distance()
            else:
                d = d_goppa
        except Exception:
            d = d_goppa

    return {
        'N': N, 'k': k, 'd': d, 'd_goppa': d_goppa,
        'pole_degree': pole_degree,
        'G_mat': G_mat, 'basis': basis, 'lc': lc,
    }


# ═══════════════════════════════════════════════════════════════════════
#  5.  CSS PARAMETER SELECTION  (FIXED — O(N) balanced search)
# ═══════════════════════════════════════════════════════════════════════

def choose_css_parameters(N, g, m, n, weights=None, strategy='product'):
    r"""
    Choose model degrees r1, r2 for the CSS construction.

    The basis is selected by model degree, but CSS validity and distance
    bounds are checked using actual pole degrees R1 and R2:
        R1 + R2 <= N + 2g - 2,
        d(C1) >= N - R1,
        d(C2^perp) >= R2 - (2g - 2).

    Strategies (all search the saturated line):
      'product'   -- maximise  k_Q * d_q_bound  (recommended)
      'balanced'  -- maximise  d_q_bound  first, then k_Q
                     (gives highest certified distance, but k_Q ~ 1)
      'max_k_q'   -- maximise  k_Q  first, then d_q_bound
                     (highest dimension, but d_q_bound ~ 1)

    OUTPUT:
      (r1, r2, d_q_bound)  or  (None, None, None)
    """
    N  = int(N)
    g  = int(g)
    m  = int(m)
    n  = int(n)
    if weights is None:
        weights = (m, n, m)

    if N <= 2:
        return (None, None, None)

    B = N + 2 * g - 2

    # r is a model degree. N is enough for the supported models because any
    # useful C1 must have actual pole degree < N.
    dims, poles = _precompute_model_data(m, n, weights, N)

    best_score = None
    best_r1    = None
    best_r2    = None
    best_dq    = None

    for r2 in range(0, N + 1):
        R2 = poles[r2]
        if R2 < 2 * g - 1:
            continue
        for r1 in range(r2 + 1, N + 1):
            R1 = poles[r1]
            if R1 >= N or R1 + R2 > B:
                continue
            k1 = dims[r1]
            k2 = dims[r2]
            k_q = k1 - k2
            if k_q <= 0:
                continue
            d1 = max(1, N - R1)
            d2d = max(1, R2 - 2 * g + 2)
            d_q = min(d1, d2d)

            if strategy == 'product':
                score = (k_q * d_q, d_q, k_q)
            elif strategy == 'balanced':
                score = (d_q, k_q)
            elif strategy == 'max_k_q':
                score = (k_q, d_q)
            else:
                score = (k_q * d_q, d_q, k_q)

            if best_score is None or score > best_score:
                best_score = score
                best_r1 = r1
                best_r2 = r2
                best_dq = d_q

    if best_r1 is not None:
        return (best_r1, best_r2, best_dq)

    return (None, None, None)


# ═══════════════════════════════════════════════════════════════════════
#  6.  BUILD CSS QUANTUM CODE
# ═══════════════════════════════════════════════════════════════════════

def build_css_quantum_code(F, f_poly, m, r1, r2, pts, g, weights=None,
                            compute_exact_d=False):
    r"""
    Build a CSS quantum code [[N, k_q, d_q]]_q from nested AG codes.
    """
    n = ZZ(f_poly.degree())
    N = len(pts)
    result = {
        'css_ok': False, 'N': N, 'k_q': 0, 'd_q': 0, 'd_q_bound': 0,
        'C1_params': (N, 0, 0), 'C2_params': (N, 0, 0),
        'skip_reason': None,
    }

    if N == 0:
        result['skip_reason'] = 'N=0'
        return result

    if weights is None:
        weights = (m, n, m)

    C1 = build_weighted_ag_code(F, f_poly, m, r1, pts, weights=weights,
                                 compute_exact_d=compute_exact_d)
    C2 = build_weighted_ag_code(F, f_poly, m, r2, pts, weights=weights,
                                 compute_exact_d=compute_exact_d)

    R1 = C1['pole_degree']
    R2 = C2['pole_degree']

    css_bound = N + 2 * g - 2
    if R1 + R2 > css_bound:
        result['skip_reason'] = (
            f'CSS_violated: pole1+pole2={R1+R2}>{css_bound}=N+2g-2'
        )
        return result

    result['C1_params'] = (C1['N'], C1['k'], C1['d'])

    result['C2_params'] = (C2['N'], C2['k'], C2['d'])

    k_q = C1['k'] - C2['k']
    if k_q <= 0:
        result['skip_reason'] = f'k_q={k_q}<=0'
        return result

    result['k_q'] = k_q

    d1_bound = max(1, N - R1)
    d2_dual_bound = max(1, R2 - 2 * g + 2)
    d_q_bound = min(d1_bound, d2_dual_bound)
    result['d_q_bound'] = d_q_bound

    d_q = d_q_bound
    is_classical = tuple(ZZ(w) for w in weights) == (1, 1, 1)
    if is_classical and C1['lc'] is not None and C2['lc'] is not None:
        try:
            if not C2['lc'].dual_code().is_subcode(C1['lc']):
                result['skip_reason'] = 'CSS_subcode_check_failed'
                return result
        except Exception:
            result['skip_reason'] = 'CSS_subcode_check_failed'
            return result

    if compute_exact_d and C2['lc'] is not None:
        try:
            C2_dual = C2['lc'].dual_code()
            if C1['lc'] is not None:
                css_verified = C2_dual.is_subcode(C1['lc'])
                if not css_verified:
                    result['skip_reason'] = 'CSS_subcode_check_failed'
                    return result
            d2_dual_exact = C2_dual.minimum_distance()
            d_q = min(C1['d'], d2_dual_exact)
        except Exception:
            d_q = d_q_bound

    result['d_q'] = d_q
    result['css_ok'] = True
    return result


# ═══════════════════════════════════════════════════════════════════════
#  7.  AUTOMORPHISM INFORMATION
# ═══════════════════════════════════════════════════════════════════════

def automorphism_lower_bound(F, m, n):
    q = F.cardinality()
    m = ZZ(m)
    has_zeta = (q - 1) % m == 0
    if has_zeta:
        return {
            'aut_lower_bound': m,
            'cyclic_order': m,
            'has_zeta_m': True,
            'aut_description': f'C_{m} (cyclic, y -> zeta_{m}*y)',
        }
    else:
        return {
            'aut_lower_bound': 1,
            'cyclic_order': 1,
            'has_zeta_m': False,
            'aut_description': f'trivial over F_{q} (zeta_{m} not in F_{q})',
        }


def automorphism_group_affine_exact(F, f_poly, m, max_generators=12):
    r"""
    Exact finite-field automorphisms for y^m=f(x) fixing P_infinity.

    For gcd(m,deg(f))=1 there is a unique point at infinity.  Hence every
    curve automorphism fixes P_infinity, and x must map to u*x+v.  Since
    char(F) does not divide m, y maps to beta*y.  We therefore test:

        f(u*x+v) = beta^m * f(x).

    Returns all automorphisms over F of this form as triples (u,v,beta).
    """
    m = ZZ(m)
    Rx = f_poly.parent()
    x = Rx.gen()
    f_poly = Rx(f_poly)

    automorphisms = []
    reduced_affine = []

    beta_by_power = {}
    for beta in F:
        beta_by_power.setdefault(beta ** m, []).append(beta)

    for u in F:
        if u == 0:
            continue
        for v in F:
            transformed = f_poly(u * x + v)
            alpha = None
            if f_poly != 0:
                lc = f_poly.leading_coefficient()
                if lc != 0:
                    alpha = transformed.leading_coefficient() / lc
            if alpha is None:
                continue
            if transformed != alpha * f_poly:
                continue

            betas = beta_by_power.get(F(alpha), [])
            if not betas:
                continue

            reduced_affine.append((u, v, F(alpha)))
            for beta in betas:
                automorphisms.append((u, v, beta))

    non_identity = [
        aut for aut in automorphisms
        if not (aut[0] == F(1) and aut[1] == F(0) and aut[2] == F(1))
    ]
    sample_source = non_identity + [
        aut for aut in automorphisms
        if aut[0] == F(1) and aut[1] == F(0) and aut[2] == F(1)
    ]

    sample = []
    for u, v, beta in sample_source[:max_generators]:
        sample.append(f"x->{u}*x+{v}; y->{beta}*y")

    return {
        'aut_exact_order': len(automorphisms),
        'reduced_affine_order': len(reduced_affine),
        'deck_order': len(beta_by_power.get(F(1), [])),
        'automorphisms': automorphisms,
        'reduced_affine': reduced_affine,
        'sample_generators': sample,
        'description': (
            f"Aut_Fq order {len(automorphisms)}; "
            f"reduced affine order {len(reduced_affine)}; "
            f"deck order {len(beta_by_power.get(F(1), []))}"
        ),
    }


# ═══════════════════════════════════════════════════════════════════════
#  8.  SINGLE CURVE RUNNER
# ═══════════════════════════════════════════════════════════════════════

def run_one_m5_curve(q, coeffs, m=5, weights=None,
                      galois_label='', compute_exact_d=False,
                      css_strategy='product', compute_aut_exact=False,
                      verbose=True):
    t0 = time.time()

    row = {
        'q': q, 'm': m, 'coeffs': str(coeffs),
        'galois_label': galois_label,
        'weights': '', 'model_label': '', 'f_x': '',
        'genus': '', 'N': '', 'r1': '', 'r2': '',
        'd1_bound': '', 'd2_dual_bound': '', 'd_q_bound': '',
        'css_strategy': css_strategy,
        'C1': '', 'C2': '', 'Q': '',
        'aut_lower_bound': '', 'aut_description': '',
        'aut_exact_order': '', 'aut_reduced_order': '',
        'aut_exact_description': '', 'aut_sample': '',
        'css_ok': False, 'skip_reason': '', 'time_s': '',
    }

    F = GF(q)
    Rx = PolynomialRing(F, 'x')
    f_coeffs_F = [F(c) for c in coeffs]
    f_poly = Rx(f_coeffs_F)
    row['f_x'] = str(f_poly)

    n_actual = f_poly.degree()
    if n_actual < 2:
        row['skip_reason'] = f'deg(f)={n_actual}<2_over_F_{q}'
        row['time_s'] = f'{time.time()-t0:.3f}'
        return row

    if weights is None:
        weights_use = (ZZ(m), ZZ(n_actual), ZZ(m))
    else:
        weights_use = tuple(ZZ(w) for w in weights)

    val = validate_superelliptic_curve(F, f_poly, m, weights_use)
    row['weights'] = str(val['weights']) if val['weights'] else ''
    row['model_label'] = val['model_label'] or ''
    row['genus'] = val['genus'] if val['genus'] is not None else ''

    if not val['ok']:
        if verbose:
            print(f"\n{'='*68}")
            print(f"Curve: y^{m} = {f_poly}   over GF({q})")
            print(f"Model: {val['model_label']}   weights={val['weights']}")
            print(f"SKIPPED: {val['skip_reason']}")
            print(f"{'='*68}")
        row['skip_reason'] = val['skip_reason']
        row['time_s'] = f'{time.time()-t0:.3f}'
        return row

    g = val['genus']
    n = val['n']

    aut = automorphism_lower_bound(F, m, n)
    row['aut_lower_bound'] = aut['aut_lower_bound']
    row['aut_description'] = aut['aut_description']

    if compute_aut_exact:
        aut_exact = automorphism_group_affine_exact(F, f_poly, m)
        row['aut_exact_order'] = aut_exact['aut_exact_order']
        row['aut_reduced_order'] = aut_exact['reduced_affine_order']
        row['aut_exact_description'] = aut_exact['description']
        row['aut_sample'] = " | ".join(aut_exact['sample_generators'])

    if verbose:
        print(f"\n{'='*68}")
        print(f"Curve: y^{m} = {f_poly}   over GF({q})")
        print(f"Model: {val['model_label']}   weights={val['weights']}")
        print(f"Genus: g = {g},  Semigroup: <{m},{n}>")
        print(f"Aut(X) >= {aut['aut_lower_bound']}")
        if compute_aut_exact:
            print(f"Aut_Fq(X): order {row['aut_exact_order']}")
            print(f"  reduced affine order: {row['aut_reduced_order']}")
            if row['aut_sample']:
                print(f"  sample: {row['aut_sample'].split(' | ')[0]}")

    pts = affine_points_superelliptic(F, f_poly, m)
    N = len(pts)
    row['N'] = N

    if verbose:
        print(f"Affine rational points: N = {N}")

    if N == 0:
        row['skip_reason'] = 'N=0'
        row['time_s'] = f'{time.time()-t0:.3f}'
        return row

    # --- choose CSS parameters for this model ---
    css_result = choose_css_parameters(N, g, m, n, weights=weights_use,
                                       strategy=css_strategy)
    r1, r2, d_q_est = css_result

    if r1 is None:
        if verbose:
            print("Quantum: SKIPPED (no_valid_css_pair)")
            print(f"{'='*68}")
        row['skip_reason'] = 'no_valid_css_pair'
        row['time_s'] = f'{time.time()-t0:.3f}'
        return row

    row['r1'] = r1
    row['r2'] = r2
    basis1_pred = rr_basis_superelliptic(m, n, r1, weights=weights_use)
    basis2_pred = rr_basis_superelliptic(m, n, r2, weights=weights_use)
    R1_pred = pole_degree_of_basis(m, n, basis1_pred)
    R2_pred = pole_degree_of_basis(m, n, basis2_pred)

    row['d1_bound'] = max(1, N - R1_pred)
    row['d2_dual_bound'] = max(1, R2_pred - 2 * g + 2)
    row['d_q_bound'] = min(row['d1_bound'], row['d2_dual_bound'])

    if verbose:
        k1_pred = len(basis1_pred)
        k2_pred = len(basis2_pred)
        print(f"CSS candidate: r1={r1}, r2={r2}  (strategy={css_strategy})")
        print(f"  pole_degree(r1)={R1_pred}, pole_degree(r2)={R2_pred}")
        print(f"  l(r1)={k1_pred}, l(r2)={k2_pred}, k_Q={k1_pred-k2_pred}")
        print(f"  d(C1)>={row['d1_bound']}, d(C2^perp)>={row['d2_dual_bound']}")
        print(f"  candidate d_Q >= {row['d_q_bound']} (requires CSS nesting)")

    C1 = build_weighted_ag_code(F, f_poly, m, r1, pts, weights=weights_use,
                                 compute_exact_d=compute_exact_d)
    C2 = build_weighted_ag_code(F, f_poly, m, r2, pts, weights=weights_use,
                                 compute_exact_d=compute_exact_d)

    row['C1'] = f"[{C1['N']},{C1['k']},{C1['d']}]"
    row['C2'] = f"[{C2['N']},{C2['k']},{C2['d']}]"

    if verbose:
        print(f"C_1 = [{C1['N']},{C1['k']},{C1['d']}]")
        print(f"C_2 = [{C2['N']},{C2['k']},{C2['d']}]")

    Q = build_css_quantum_code(F, f_poly, m, r1, r2, pts, g,
                                weights=weights_use,
                                compute_exact_d=compute_exact_d)

    if Q['css_ok']:
        row['Q'] = f"[[{Q['N']},{Q['k_q']},{Q['d_q']}]]"
        row['d_q_bound'] = Q['d_q_bound']
        row['css_ok'] = True
    else:
        row['Q'] = ''
        row['skip_reason'] = Q.get('skip_reason', 'css_failed')

    if verbose:
        if Q['css_ok']:
            print(f"Quantum: [[{Q['N']},{Q['k_q']},{Q['d_q']}]]_{q}")
        else:
            print(f"Quantum: SKIPPED ({Q.get('skip_reason','')})")

    if verbose:
        print(f"{'='*68}")

    row['time_s'] = f'{time.time()-t0:.3f}'
    return row


# ═══════════════════════════════════════════════════════════════════════
#  9.  DATABASE PROCESSOR
# ═══════════════════════════════════════════════════════════════════════

def _parse_aims7_columns(df_row, col_names):
    coeffs_raw = None
    for cname in ['coeffs',
                  '=HYPERLINK("https://www.lmfdb.org/knowledge/show/nf.defining_polynomial", "coeffs")']:
        if cname in col_names:
            coeffs_raw = df_row.get(cname, None)
            break
    if coeffs_raw is None:
        keys = list(df_row.keys())
        if len(keys) >= 2:
            coeffs_raw = df_row[keys[1]]

    gal_label = None
    for cname in ['galois_label',
                  '=HYPERLINK("https://www.lmfdb.org/knowledge/show/nf.galois_group", "galois_label")']:
        if cname in col_names:
            gal_label = df_row.get(cname, '')
            break
    if gal_label is None:
        keys = list(df_row.keys())
        gal_label = df_row[keys[3]] if len(keys) >= 4 else ''

    label = None
    for cname in ['label',
                  '=HYPERLINK("https://www.lmfdb.org/knowledge/show/nf.label", "label")']:
        if cname in col_names:
            label = df_row.get(cname, '')
            break
    if label is None:
        keys = list(df_row.keys())
        label = df_row.get(keys[0], '') if keys else ''

    math_gal = df_row.get('math_galois_notation', '')
    return coeffs_raw, gal_label, label, math_gal


def process_m5_database(csv_path, output_csv, q=11, m=5,
                         weights=None, limit=None,
                         compute_exact_d=False,
                         compute_aut_exact=False,
                         css_strategy='product',
                         verbose_every=50000):
    r"""
    Process the AIMS-7 database. Writes results INCREMENTALLY
    to output_csv so partial results survive crashes.
    """
    print(f"\n{'#'*68}")
    print(f"# Codex Classic Codes: m={m} superelliptic AG/CSS pipeline")
    print(f"# Field: GF({q})")
    print(f"# Weights: {weights if weights else f'natural P({m},7,{m})'}")
    print(f"# CSS strategy: {css_strategy}")
    print(f"# Input:  {csv_path}")
    print(f"# Output: {output_csv}")
    print(f"{'#'*68}\n")

    rows_in = []
    with open(csv_path, 'r', encoding='utf-8') as fh:
        reader = _csv.DictReader(fh)
        col_names = reader.fieldnames
        for row in reader:
            rows_in.append(row)

    total = len(rows_in)
    print(f"Loaded {total} rows from database")
    if limit:
        rows_in = rows_in[:limit]
        print(f"Processing first {limit}")

    out_fields = [
        'label', 'galois_label', 'math_galois', 'coeffs',
        'q', 'm', 'weights', 'model_label', 'f_x',
        'genus', 'N', 'r1', 'r2',
        'd1_bound', 'd2_dual_bound', 'd_q_bound', 'css_strategy',
        'C1', 'C2', 'Q',
        'aut_lower_bound', 'aut_description',
        'aut_exact_order', 'aut_reduced_order',
        'aut_exact_description', 'aut_sample',
        'css_ok', 'skip_reason', 'time_s',
    ]

    stats = {
        'total': len(rows_in), 'valid': 0,
        'ag_ok': 0, 'quantum_ok': 0, 'skipped': {},
    }

    t_start = time.time()

    # --- INCREMENTAL WRITE: open file once, flush regularly ---
    fh_out = open(output_csv, 'w', newline='', encoding='utf-8')
    writer = _csv.DictWriter(fh_out, fieldnames=out_fields,
                              extrasaction='ignore')
    writer.writeheader()

    for idx, raw_row in enumerate(rows_in):
        coeffs_raw, gal_label, label, math_gal = \
            _parse_aims7_columns(raw_row, col_names)

        try:
            coeffs = list(ast.literal_eval(str(coeffs_raw).strip()))
        except Exception:
            stats['skipped']['parse_error'] = \
                stats['skipped'].get('parse_error', 0) + 1
            continue

        row_result = run_one_m5_curve(
            q=q, coeffs=coeffs, m=m, weights=weights,
            galois_label=gal_label,
            compute_exact_d=compute_exact_d,
            compute_aut_exact=compute_aut_exact,
            css_strategy=css_strategy,
            verbose=False,
        )

        row_result['label'] = label
        row_result['math_galois'] = math_gal

        # Write immediately
        writer.writerow(row_result)

        if row_result.get('C1', ''):
            stats['ag_ok'] += 1

        if row_result.get('css_ok', False):
            stats['quantum_ok'] += 1

        if row_result.get('skip_reason', ''):
            reason = row_result['skip_reason']
            stats['skipped'][reason] = stats['skipped'].get(reason, 0) + 1
        else:
            stats['valid'] += 1

        if (idx + 1) % verbose_every == 0:
            fh_out.flush()
            elapsed = time.time() - t_start
            rate = (idx + 1) / elapsed if elapsed > 0 else 0
            eta = (len(rows_in) - idx - 1) / rate if rate > 0 else 0
            print(f"  [{idx+1}/{len(rows_in)}]  "
                  f"AG={stats['ag_ok']}  Q={stats['quantum_ok']}  "
                  f"skip={sum(stats['skipped'].values())}  "
                  f"({rate:.1f} rows/s, ETA {eta/60:.0f}min)")

    fh_out.close()

    elapsed_total = time.time() - t_start
    print(f"\n{'='*68}")
    print(f"DONE in {elapsed_total:.1f}s ({elapsed_total/60:.1f} min)")
    print(f"  Total rows:    {stats['total']}")
    print(f"  Valid quantum rows: {stats['valid']}")
    print(f"  Classical AG rows:  {stats['ag_ok']}")
    print(f"  Quantum codes:      {stats['quantum_ok']}")
    print(f"  Skipped:       {sum(stats['skipped'].values())}")
    for reason, count in sorted(stats['skipped'].items(),
                                 key=lambda x: -x[1]):
        print(f"    {reason}: {count}")
    print(f"  Output -> {output_csv}")
    print(f"{'='*68}")

    return stats


# ═══════════════════════════════════════════════════════════════════════
#  10.  EXAMPLE / SELF-TEST
# ═══════════════════════════════════════════════════════════════════════

def run_example():
    print("\n" + "="*68)
    print("  SELF-TEST: balanced CSS on y^5 = f(x)")
    print("="*68)

    test_polys = [
        ([1, 0, 1, -1, -1, 0, 0, 1], '7T7', 'S7'),   # N=35 over F_11
        ([1, 1, -1, 0, 1, -1, -1, 1], '7T7', 'S7'),   # N=15 over F_11
    ]

    fields = [11, 31, 61]
    weight_choices = [(5,7,5)]

    for coeffs, gal, _ in test_polys:
        for q in fields:
            for w in weight_choices:
                row = run_one_m5_curve(
                    q=q, coeffs=coeffs, m=5, weights=w,
                    galois_label=gal,
                    css_strategy='product',
                    compute_aut_exact=True,
                    verbose=True,
                )

    print("\n" + "="*68)
    print("  Self-test complete.")
    print("="*68)


def _running_pipeline_file_directly():
    r"""
    Sage's load(...) executes files with __name__ == '__main__'.  Use argv[0]
    to distinguish direct self-test runs from database runner loads.
    """
    top_script = os.path.basename(str(sys.argv[0])).replace('.sage.py', '.sage')
    return top_script == 'codex_classic_superelliptic.sage'


if __name__ == '__main__' and _running_pipeline_file_directly():
    run_example()


In [ ]:
#!/usr/bin/env sage
r"""
Run Codex Classic superelliptic AG/CSS codes over GF(61).

Output:
  codex_classic_results/codex_classic_q61.csv
"""

import os, sys, time


def find_base_dir():
    candidates = []

    argv_dir = os.path.dirname(os.path.abspath(str(sys.argv[0])))
    if argv_dir.endswith('.sage.py'):
        argv_dir = os.path.dirname(argv_dir)
    candidates.append(argv_dir)

    candidates.append("/Users/jurimezini/Downloads/codex-classic-codes")
    candidates.append(os.getcwd())

    for path in candidates:
        pipeline = os.path.join(path, "codex_classic_superelliptic.sage")
        if os.path.exists(pipeline):
            return path

    raise RuntimeError("Could not find codex_classic_superelliptic.sage")


BASE_DIR = find_base_dir()
load(os.path.join(BASE_DIR, "codex_classic_superelliptic.sage"))

CSV_PATH = "/Users/jurimezini/Library/CloudStorage/Dropbox/Sage_Galois7/AIMS-7.csv"
OUTPUT_DIR = os.path.join(BASE_DIR, "codex_classic_results")
os.makedirs(OUTPUT_DIR, exist_ok=True)

Q = 61
CSS_STRATEGY = "product"
TEST_LIMIT = None
VERBOSE_EVERY = 50000

out_csv = os.path.join(OUTPUT_DIR, f"codex_classic_q{Q}.csv")

print("\n" + "=" * 70)
print(f"  Codex Classic Codes over GF({Q})")
print("=" * 70)
print(f"  Output -> {out_csv}")

t0 = time.time()
stats = process_m5_database(
    csv_path=CSV_PATH,
    output_csv=out_csv,
    q=Q,
    m=5,
    weights=(5, 7, 5),
    limit=TEST_LIMIT,
    compute_exact_d=False,
    css_strategy=CSS_STRATEGY,
    verbose_every=VERBOSE_EVERY,
)

elapsed = time.time() - t0
print(f"\n{'='*70}")
print(f"  Codex Classic GF({Q}) complete in {elapsed/60:.1f} minutes")
print(f"  Output: {out_csv}")
print(f"{'='*70}")


In [ ]:
#!/usr/bin/env sage
r"""
Run Codex Classic superelliptic AG/CSS codes over GF(11).

Output:
  codex_classic_results/codex_classic_q11.csv
"""

import os, sys, time


def find_base_dir():
    candidates = []

    argv_dir = os.path.dirname(os.path.abspath(str(sys.argv[0])))
    if argv_dir.endswith('.sage.py'):
        argv_dir = os.path.dirname(argv_dir)
    candidates.append(argv_dir)

    candidates.append("/Users/jurimezini/Downloads/codex-classic-codes")
    candidates.append(os.getcwd())

    for path in candidates:
        pipeline = os.path.join(path, "codex_classic_superelliptic.sage")
        if os.path.exists(pipeline):
            return path

    raise RuntimeError("Could not find codex_classic_superelliptic.sage")


BASE_DIR = find_base_dir()
load(os.path.join(BASE_DIR, "codex_classic_superelliptic.sage"))

CSV_PATH = "/Users/jurimezini/Library/CloudStorage/Dropbox/Sage_Galois7/AIMS-7.csv"
OUTPUT_DIR = os.path.join(BASE_DIR, "codex_classic_results")
os.makedirs(OUTPUT_DIR, exist_ok=True)

Q = 11
CSS_STRATEGY = "product"
TEST_LIMIT = None
VERBOSE_EVERY = 50000

out_csv = os.path.join(OUTPUT_DIR, f"codex_classic_q{Q}.csv")

print("\n" + "=" * 70)
print(f"  Codex Classic Codes over GF({Q})")
print("=" * 70)
print(f"  Output -> {out_csv}")

t0 = time.time()
stats = process_m5_database(
    csv_path=CSV_PATH,
    output_csv=out_csv,
    q=Q,
    m=5,
    weights=(5, 7, 5),
    limit=TEST_LIMIT,
    compute_exact_d=False,
    css_strategy=CSS_STRATEGY,
    verbose_every=VERBOSE_EVERY,
)

elapsed = time.time() - t0
print(f"\n{'='*70}")
print(f"  Codex Classic GF({Q}) complete in {elapsed/60:.1f} minutes")
print(f"  Output: {out_csv}")
print(f"{'='*70}")
